In [ ]:
import torch
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
project_root = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
load_dotenv(dotenv_path=project_root / ".env", override=False)

# 1) Read concept bank path from .env (or construct from OUTPUT_DIR)
# Priority: CONCEPT_PATH > OUTPUT_DIR/concept/snmf/combined_concept_snmf_raw.pth
CONCEPT_PATH = os.getenv("CONCEPT_PATH")
OUTPUT_DIR = os.getenv("OUTPUT_DIR", str(project_root / "outputs/noidle"))

if CONCEPT_PATH:
    file_path = CONCEPT_PATH
else:
    # Default: OUTPUT_DIR/concept/snmf/combined_concept_snmf_raw.pth
    file_path = os.path.join(OUTPUT_DIR, "concept", "snmf", "combined_concept_snmf_raw.pth")

# 2) Read IMAGE_SIZE_WIDTH for visualization (default: 512)
IMAGE_SIZE_WIDTH = int(os.getenv("IMAGE_SIZE_WIDTH", "512"))

print(f"CONCEPT_PATH: {file_path}")
print(f"IMAGE_SIZE_WIDTH: {IMAGE_SIZE_WIDTH}")

concept_pth = Path(file_path)
assert concept_pth.exists(), f"Concept bank not found: {concept_pth}"
assert "raw" in concept_pth.stem.lower(), f"Expected a raw concept file, got: {concept_pth.name}"

data1 = torch.load(str(concept_pth), map_location="cpu")
print("Concept keys:", list(data1.keys()))

# Lightweight sanity print (avoid dumping huge arrays)
for k in ["concepts", "activations", "text_grounding", "image_grounding_paths", "image_grounding_bboxes", "image_grounding_predictions"]:
    if k not in data1:
        continue
    v = data1[k]
    if isinstance(v, torch.Tensor):
        print(f"{k}: tensor shape={tuple(v.shape)} dtype={v.dtype}")
    else:
        try:
            print(f"{k}: type={type(v).__name__} len={len(v)}")
        except Exception:
            print(f"{k}: type={type(v).__name__}")

# Peek at one concept's metadata (if present)
idx0 = 0
if isinstance(data1.get("text_grounding"), (list, tuple)) and len(data1["text_grounding"]) > 0:
    print("text_grounding[0] =", data1["text_grounding"][idx0])
if isinstance(data1.get("image_grounding_paths"), (list, tuple)) and len(data1["image_grounding_paths"]) > 0:
    print("image_grounding_paths[0] =", data1["image_grounding_paths"][idx0])
if isinstance(data1.get("image_grounding_bboxes"), (list, tuple)) and len(data1["image_grounding_bboxes"]) > 0:
    print("image_grounding_bboxes[0] =", data1["image_grounding_bboxes"][idx0])

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

# 2) Inputs
# Single-image mode:
image_to_infer = "/mnt/abka03/xlvlm_data/imagenet_3_class/train/cat/n02123045_10633.JPEG"

# Multi-image mode (uncomment and add more paths):
# images_to_infer = [
#     "/path/to/img1.jpg",
#     "/path/to/img2.jpg",
# ]

word_to_explain = "eye"

# Build a binary prompt. The explainer will also enforce a strict binary instruction.
#prompt = f"Is there a '{word_to_explain}' in the image? Answer '{word_to_explain}' or 'no {word_to_explain}'."
prompt = f"Is there a '{word_to_explain}' in the image? Write '{word_to_explain}' if it exist in the image otherwise nothing. "
# 3) Load config (prefer .env if present; otherwise defaults match api/main.py)
project_root = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
load_dotenv(dotenv_path=project_root / ".env", override=False)

VLM_MODEL = os.getenv("VLM_MODEL", "Qwen/Qwen2.5-VL-3B-Instruct")
LAYER_PATH = os.getenv("LAYER_PATH", "model.language_model.norm")

# Resolve images list + choose multibatch when multiple images are provided
if "images_to_infer" in globals() and images_to_infer is not None:
    images = list(images_to_infer)
else:
    images = [image_to_infer]

images = [str(p) for p in images]
use_multibatch = len(images) > 1
print("VLM_MODEL:", VLM_MODEL)
print("LAYER_PATH:", LAYER_PATH)
print("#images:", len(images), "multibatch:", use_multibatch)
print("prompt:", prompt)


In [ ]:
# 3.5) Sanity import (no model load)
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(project_root / "inference"))

from vlm_explainer_multibatch import VLMConceptExplainer
print("Imported VLMConceptExplainer:", VLMConceptExplainer)


In [ ]:
# 4) Inference + hook + concept scoring
# This cell implements your comments:
# - apply hook at LAYER_PATH
# - extract residual/hidden embeddings for generated output tokens
# - compute cosine similarity to concept bank

import sys
from pathlib import Path

# Ensure we can import the multibatch explainer
project_root = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(project_root / "inference"))

from vlm_explainer_multibatch import VLMConceptExplainer

# Use multibatch only when multiple images are given
batch_size = min(2, len(images)) if len(images) > 1 else 1

explainer = VLMConceptExplainer(
    model_name=VLM_MODEL,
    concept_path=str(concept_pth),
    layer_path=LAYER_PATH,
    prompt_mode="binary",
    prompt_label=word_to_explain,
    # keep raw concepts; cosine similarity is scale-invariant
    normalize_concepts=False,
    verbose=False,
    # helps align activations to generated tokens
    save_only_generated_tokens=True,
)

results = explainer.explain_with_concept(
    images=images,
    ground_truth_labels=[word_to_explain] * len(images),
    top_n=5,
    max_new_tokens=80,
    temperature=0.0,
    batch_size=batch_size,
)
explainer.close()

print("Got", len(results), "result(s)")
print("Example keys:", list(results[0].keys()))
print("Model output[0]:", results[0]["model_output"])
print("Top concepts (aggregate)[0]:")
for c in (results[0].get("top_concepts_over_sequence") or [])[:5]:
    print(" ", f"#{c['rank']} idx={c['concept_index']} sim={c['similarity']:.4f} text={c.get('text_grounding')}")


In [ ]:
# 5) Prototype + bbox + segmentation mask visualization for TOP 3 concepts
#
# NOTE: image_grounding_bboxes[concept] is the bbox ON image_grounding_paths[concept],
#       not on the input image. So we draw the bbox on the prototype image.
# NOTE: image_grounding_masks[concept] is a list of RLE-encoded segmentation masks
#       (pycocotools format) parallel to image_grounding_paths. We decode and overlay them.
#
# Uses IMAGE_SIZE_WIDTH from .env for consistent visualization sizing.

import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt


def _pick_first_bbox(b):
    """Return a single [x1,y1,x2,y2] bbox if present; else None."""
    if b is None:
        return None
    if isinstance(b, (list, tuple)) and len(b) == 4 and all(isinstance(x, (int, float)) for x in b):
        return list(map(float, b))
    if isinstance(b, (list, tuple)) and len(b) > 0 and isinstance(b[0], (list, tuple)):
        return _pick_first_bbox(b[0])
    return None


def _resolve_proto_path(p: str) -> Path:
    pp = Path(str(p))
    if pp.exists():
        return pp
    alt = concept_pth.parent / pp
    return alt


def resize_to_width(img: Image.Image, target_width: int) -> Image.Image:
    """Resize image to target width while maintaining aspect ratio."""
    if target_width <= 0:
        return img
    w, h = img.size
    if w == target_width:
        return img
    ratio = target_width / w
    new_h = int(h * ratio)
    return img.resize((target_width, new_h), Image.Resampling.LANCZOS)


def draw_bbox_on_image(img: Image.Image, bbox, color="red", width=4) -> Image.Image:
    """Draw a bbox [x1,y1,x2,y2] on an image copy and return it."""
    if bbox is None:
        return img
    img_draw = img.copy()
    draw = ImageDraw.Draw(img_draw)
    x1, y1, x2, y2 = bbox
    draw.rectangle([x1, y1, x2, y2], outline=color, width=width)
    return img_draw


def decode_mask_rle(rle_dict):
    """Decode an RLE mask dict (pycocotools format) to a boolean numpy array."""
    if rle_dict is None or isinstance(rle_dict, str):
        return None
    if not isinstance(rle_dict, dict) or "counts" not in rle_dict:
        return None
    try:
        import pycocotools.mask as mask_util
        rle = dict(rle_dict)
        if isinstance(rle["counts"], str):
            rle["counts"] = rle["counts"].encode("utf-8")
        return mask_util.decode(rle).astype(bool)
    except Exception:
        return None


def overlay_mask_on_image(img: Image.Image, mask: np.ndarray, color=(0, 255, 0), alpha=0.4) -> Image.Image:
    """Overlay a boolean segmentation mask on an image with a semi-transparent color."""
    if mask is None:
        return img
    img_np = np.array(img).copy()
    h_img, w_img = img_np.shape[:2]
    h_mask, w_mask = mask.shape[:2]

    # Resize mask to match image dimensions if needed
    if (h_mask, w_mask) != (h_img, w_img):
        from PIL import Image as PILImage
        mask_pil = PILImage.fromarray(mask.astype(np.uint8) * 255)
        mask_pil = mask_pil.resize((w_img, h_img), PILImage.Resampling.NEAREST)
        mask = np.array(mask_pil) > 127

    # Apply colored overlay where mask is True
    overlay = np.array(color, dtype=np.float32)
    for c in range(3):
        img_np[:, :, c] = np.where(
            mask,
            (1 - alpha) * img_np[:, :, c] + alpha * overlay[c],
            img_np[:, :, c]
        )
    return Image.fromarray(img_np.astype(np.uint8))


def draw_mask_contour(img: Image.Image, mask: np.ndarray, color=(0, 255, 0), width=2) -> Image.Image:
    """Draw the contour of a segmentation mask on the image."""
    if mask is None:
        return img
    import cv2
    img_np = np.array(img).copy()
    h_img, w_img = img_np.shape[:2]
    h_mask, w_mask = mask.shape[:2]

    if (h_mask, w_mask) != (h_img, w_img):
        mask_resized = cv2.resize(mask.astype(np.uint8), (w_img, h_img), interpolation=cv2.INTER_NEAREST)
    else:
        mask_resized = mask.astype(np.uint8)

    contours, _ = cv2.findContours(mask_resized, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(img_np, contours, -1, color, width)
    return Image.fromarray(img_np)


# Mask overlay color palette (one color per concept rank)
MASK_COLORS = [
    (0, 255, 100),   # green
    (255, 100, 0),   # orange
    (100, 100, 255), # blue
    (255, 255, 0),   # yellow
    (255, 0, 255),   # magenta
]


# --- Visualize the first result ---
r0 = results[0]
img0_path = images[0]
img_input = Image.open(img0_path).convert("RGB")

# Resize input image to IMAGE_SIZE_WIDTH for visualization
img_input_resized = resize_to_width(img_input, IMAGE_SIZE_WIDTH)

# Calculate figure size based on IMAGE_SIZE_WIDTH (scale proportionally)
fig_scale = IMAGE_SIZE_WIDTH / 512.0
input_fig_width = max(6, 8 * fig_scale)
input_fig_height = max(4, 6 * fig_scale)

# --- 1) Show the INPUT image ---
plt.figure(figsize=(input_fig_width, input_fig_height))
plt.title(f"Input image | word='{word_to_explain}' | model_output='{r0['model_output'].strip()}'")
plt.imshow(img_input_resized)
plt.axis("off")
plt.show()
print(f" I learnt how {word_to_explain} look like and what other objects it relates to, for example.")
print(f" (Image displayed at width={IMAGE_SIZE_WIDTH}px)")

# --- 2) Show TOP 3 concepts, each with ALL 5 prototype images ---
# Each prototype shows: original image + bbox (green) + segmentation mask overlay (colored)
top_concepts = r0.get("top_concepts_over_sequence") or []
n_concepts_to_show = min(3, len(top_concepts))
n_protos_to_show = 5

proto_fig_width_per_img = max(3, 4 * fig_scale)

for rank, concept_info in enumerate(top_concepts[:n_concepts_to_show], start=1):
    concept_idx = int(concept_info["concept_index"])
    text_grounding = concept_info.get("text_grounding")
    similarity = concept_info.get("similarity", 0.0)
    mask_color = MASK_COLORS[(rank - 1) % len(MASK_COLORS)]

    # Get prototype paths, bboxes, and masks
    proto = concept_info.get("image_grounding_path")
    proto_list = proto if isinstance(proto, (list, tuple)) else ([proto] if proto else [])
    proto_list = [p for p in proto_list if p]

    bbox_raw = concept_info.get("image_grounding_bboxes")
    if bbox_raw is None:
        bbox_list = []
    elif isinstance(bbox_raw, (list, tuple)) and len(bbox_raw) > 0 and isinstance(bbox_raw[0], (list, tuple)):
        bbox_list = bbox_raw
    else:
        bbox_list = [bbox_raw]

    mask_raw = concept_info.get("image_grounding_masks")
    if mask_raw is None:
        mask_list = []
    elif isinstance(mask_raw, (list, tuple)):
        mask_list = mask_raw
    else:
        mask_list = [mask_raw]

    if not proto_list:
        print(f"Concept #{rank} (idx={concept_idx}, text={text_grounding}): No prototype images.")
        continue

    # Count how many prototypes have valid masks
    has_any_mask = any(
        isinstance(m, dict) and "counts" in m
        for m in mask_list
    )

    n_show = min(n_protos_to_show, len(proto_list))
    # If masks available, show 2 rows: top=original+bbox, bottom=mask overlay
    n_rows = 2 if has_any_mask else 1
    fig, axes = plt.subplots(
        n_rows, n_show,
        figsize=(proto_fig_width_per_img * n_show, proto_fig_width_per_img * n_rows)
    )
    if n_rows == 1 and n_show == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes[np.newaxis, :]
    elif n_show == 1:
        axes = axes[:, np.newaxis]

    for i in range(n_show):
        if i < len(proto_list):
            pp = _resolve_proto_path(proto_list[i])
            if pp.exists():
                proto_img = Image.open(pp).convert("RGB")
                orig_w = proto_img.size[0]
                proto_img = resize_to_width(proto_img, IMAGE_SIZE_WIDTH)

                # --- Row 0: original + bbox ---
                bbox_i = bbox_list[i] if i < len(bbox_list) else None
                bbox_i = _pick_first_bbox(bbox_i)
                if bbox_i is not None:
                    scale = IMAGE_SIZE_WIDTH / orig_w if orig_w > 0 else 1.0
                    bbox_i = [coord * scale for coord in bbox_i]
                img_with_bbox = draw_bbox_on_image(proto_img, bbox_i, color="lime", width=3)
                axes[0, i].imshow(img_with_bbox)
                axes[0, i].set_title(f"Proto {i+1}", fontsize=9)
                axes[0, i].axis("off")

                # --- Row 1: mask overlay (if masks row exists) ---
                if n_rows == 2:
                    mask_i_raw = mask_list[i] if i < len(mask_list) else None
                    mask_decoded = decode_mask_rle(mask_i_raw)
                    if mask_decoded is not None:
                        img_masked = overlay_mask_on_image(proto_img.copy(), mask_decoded, color=mask_color, alpha=0.45)
                        img_masked = draw_mask_contour(img_masked, mask_decoded, color=mask_color, width=2)
                        img_masked = draw_bbox_on_image(img_masked, bbox_i, color="lime", width=2)
                        axes[1, i].imshow(img_masked)
                        axes[1, i].set_title("Mask overlay", fontsize=9)
                    else:
                        axes[1, i].imshow(proto_img)
                        axes[1, i].set_title("No mask", fontsize=9, color="gray")
                    axes[1, i].axis("off")
            else:
                axes[0, i].text(0.1, 0.5, f"Missing:\n{pp.name}", fontsize=8)
                axes[0, i].axis("off")
                if n_rows == 2:
                    axes[1, i].axis("off")
        else:
            axes[0, i].axis("off")
            if n_rows == 2:
                axes[1, i].axis("off")

    mask_label = " | ✓ seg masks" if has_any_mask else ""
    fig.suptitle(
        f"Concept #{rank}: idx={concept_idx} | sim={similarity:.4f} | text={text_grounding}{mask_label}",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()

print("\n--- Summary ---")
print(f"(All images resized to width={IMAGE_SIZE_WIDTH}px for visualization)")
for rank, c in enumerate(top_concepts[:n_concepts_to_show], start=1):
    masks = c.get("image_grounding_masks") or []
    n_masks = sum(1 for m in masks if isinstance(m, dict) and "counts" in m) if isinstance(masks, list) else 0
    mask_info = f" | masks={n_masks}" if n_masks > 0 else " | no masks"
    print(f"  #{rank} idx={c['concept_index']} sim={c.get('similarity', 0):.4f} text={c.get('text_grounding')}{mask_info}")